In [1]:
import time
import ctypes
import numpy as np
from scipy.signal import resample_poly
from scipy.io.wavfile import write as wavwrite
from pynq.overlays.base import BaseOverlay
from pynq.lib import MicroblazeLibrary
from pynq import allocate
from multiprocessing import Process, Manager, Queue
from itertools import takewhile
base = BaseOverlay("base.bit")

In [2]:
%%microblaze base.PMODA
#include <i2c.h>
#include <time.h>
#include <yield.h>
#include <pyprintf.h>

#define DIGILENT_SCL 2
#define DIGILENT_SDA 3
#define CH_0 16
#define CH_1 32
#define CH_0_1 48

#define MAX_SAMPLES 4096

size_t read_idx;
size_t write_idx;
static int samples[MAX_SAMPLES];    

i2c device;

int init_adc() {
    device = i2c_open(DIGILENT_SDA, DIGILENT_SCL);
    unsigned char buf[2];
    buf[0] = CH_0;
    i2c_write(device, 0x28, buf, 1);
    return 0;
}

int read_adc() {
    unsigned char buf[2];
    i2c_read(device, 0x28, buf, 2);
    return ((buf[0] & 0x0F) << 8) | buf[1];
}

int read_adc_multi(size_t size) {
    size_t i;
    unsigned char r[2];
    for (i = 0; i < size; ++i) {
        i2c_read(device, 0x28, r, 2);
        
        samples[i] = ((r[0] & 0x0F) << 8) | r[1];
    }
    
    return 0;
}

int read_adc_dma(void *buf, size_t size) {
    size_t i;
    int* int_data = (int*)buf;
    unsigned char r[2];
    for (i = 0; i < size; ++i) {
        i2c_read(device, 0x28, r, 2);
        
        int_data[i] = ((r[0] & 0x0F) << 8) | r[1];
    }
    return 0;
}

// writes samples to the circular buffer
void update_circular() {
    unsigned char r[2];
    while (1) {
        while (((write_idx + 1) % MAX_SAMPLES)
               != (read_idx % MAX_SAMPLES)) {
            i2c_read(device, 0x28, r, 2);
            samples[write_idx] = ((r[0] & 0x0F) << 8) | r[1];
            write_idx = ++write_idx % MAX_SAMPLES;
        }
        yield();
    }
}

int get_circular() {
    int save = 0;
    if (read_idx == write_idx)
    {
        // buffer is empty
        return -1;
    }
    
    save = samples[read_idx++];
    read_idx %= MAX_SAMPLES;
    return save;
}

int get_sample(size_t i) {
    return samples[i];
}

In [3]:
init_adc()

0

In [4]:
def despike(audio, threshold=3.0):
    mean = np.mean(audio)
    std = np.std(audio)
    spikes = np.abs(audio - mean) > threshold * std
    indices = np.arange(len(audio))
    audio[spikes] = np.interp(indices[spikes], indices[~spikes], audio[~spikes])
    return audio

def normalize(audio):
    peak = np.max(np.abs(audio))
    if peak == 0:
        return audio
    return audio / peak * 32767.0

In [11]:
CHUNK = 4096
processed = []
shared_buf = allocate(shape=(CHUNK,), dtype=np.int32)
shared_buf[:] = [0] * CHUNK

In [12]:
total_samples = 0
total_time = 0.0

while len(processed) < 200_000:
    sample_begin = time.time()
    read_adc_dma(shared_buf, CHUNK)
    sample_end = time.time()
    
    total_time += sample_end - sample_begin
    total_samples += CHUNK
    
    chunk = shared_buf[:CHUNK].astype(np.float32)
    upsampled = resample_poly(chunk, 4, 1).astype(np.int16)
    processed.extend(upsampled)

sample_rate = total_samples / total_time

In [13]:
audio = np.array(processed, dtype=np.float32)

audio -= np.mean(audio)

audio = despike(audio, threshold=3.0)
audio = normalize(audio)

wavwrite("littlesound.wav", int(sample_rate * 4), audio.astype(np.int16))

In [14]:
sample_rate

3324.3137684899243